In [1]:
import pandas as pd
import numpy as np
from torch import neg
from operator import pos

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.model_selection import cross_val_score, GridSearchCV, StratifiedKFold, cross_validate
from sklearn.naive_bayes import GaussianNB as nb
from xgboost import XGBClassifier
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score, average_precision_score, roc_auc_score, log_loss
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import TunedThresholdClassifierCV

import matplotlib.pyplot as plt
from plotly.subplots import make_subplots

In [2]:
original_train_dataset = pd.read_csv("conversion_data_train.csv")
original_train_dataset = original_train_dataset[original_train_dataset["age"]<= 100]

cvm_train_dataset = original_train_dataset.copy()

In [3]:
X = cvm_train_dataset.drop(columns=["converted"])
y = cvm_train_dataset["converted"]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# III/ Cross validation des entrainements des modèles Logistic Regression, HistGradientBoostingClassifier et XGBoost afin de trouver les meilleurs hyperparamètres et seuils pour chaque modèle.

In [5]:
cv = StratifiedKFold( 
    n_splits=5,
    shuffle=True,
    random_state=42
)

#### Modèle 1 : HistGradientBoostingClassifier

In [6]:
# preprocessing HistGradientBoostingClassifier

numeric_features = ["age", "total_pages_visited"]
categorical_features = ["country", "source", "new_user"]

categorical_transformer = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [7]:
# Gridsearch 1 qui teste l'apprentissage du modèle : 
params_hgbc_gs1 = {
    "model__learning_rate": [0.01, 0.1, 0.3, 0.5],
    "model__max_iter": [100, 300, 500],
}

# Gridsearch 2 qui teste la profondeur et le surapprentissage du modèle :
params_hgbc_gs2 = {
    "model__max_depth": [1, 3, 5, 7],
    "model__min_samples_leaf": [1, 5, 10],
    "model__l2_regularization": [0.0, 0.5, 1.0],
}

# Gridsearch 3 qui teste l'interaction entre les hyperparamètres du modèle :
params_hgbc_gs3 = {
    "model__learning_rate": [0.01, 0.1, 0.3, 0.5],
    "model__max_iter": [100, 300, 500],
    "model__max_depth": [1, 3, 5, 7],
    "model__min_samples_leaf": [1, 5, 10],
    "model__l2_regularization": [0.0, 0.5, 1.0],
}

In [8]:
model = HistGradientBoostingClassifier(random_state=42)

pipeline_hgbc = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model),
])

scoring = {
    "f1": "f1",
    "recall": "recall",
    "pr_auc": "average_precision",
    "log_loss": "neg_log_loss",
}

In [9]:
grid_hgbc_gs1 = GridSearchCV(
    estimator=pipeline_hgbc,
    param_grid=params_hgbc_gs1,
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)
grid_hgbc_gs1.fit(X_train, y_train)

grid_hgbc_gs2 = GridSearchCV(
    estimator=pipeline_hgbc,
    param_grid=params_hgbc_gs2,
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)
grid_hgbc_gs2.fit(X_train, y_train)

grid_hgbc_gs3 = GridSearchCV(
    estimator=pipeline_hgbc,
    param_grid=params_hgbc_gs3,
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)
grid_hgbc_gs3.fit(X_train, y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         'passthrough',
                                                                         ['age',
                                                                          'total_pages_visited']),
                                                                        ('cat',
                                                                         OneHotEncoder(handle_unknown='ignore',
                                                                                       sparse_output=False),
                                                                         ['country',
                                                                          'source',
                                                                          'new_user'])])),
                                       ('model',
                                        HistGradientBoostingClassifier(random_state=42))]),
             n_jobs=-1,
             param_grid={'model__l2_regularization': [0.0, 0.5, 1.0],
                         'model__learning_rate': [0.01, 0.1, 0.3, 0.5],
                         'model__max_depth': [1, 3, 5, 7],
                         'model__max_iter': [100, 300, 500],
                         'model__min_samples_leaf': [1, 5, 10]},
             refit='f1', return_train_score=True,
             scoring={'f1': 'f1', 'log_loss': 'neg_log_loss',
                      'pr_auc': 'average_precision', 'recall': 'recall'})

In [10]:
results_hgbc_gs1 = pd.DataFrame(grid_hgbc_gs1.cv_results_)
display(results_hgbc_gs1.head(10))

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__learning_rate,param_model__max_iter,params,split0_test_f1,split1_test_f1,split2_test_f1,...,mean_test_log_loss,std_test_log_loss,rank_test_log_loss,split0_train_log_loss,split1_train_log_loss,split2_train_log_loss,split3_train_log_loss,split4_train_log_loss,mean_train_log_loss,std_train_log_loss
0,3.727680,0.081667,0.677894,0.029321,0.01,100,"{'model__learning_rate': 0.01, 'model__max_ite...",0.695764,0.694600,0.695283,...,-0.056851,0.000408,6,-0.056222,-0.056448,-0.055999,-0.056169,-0.056307,-0.056229,0.000149
1,9.599144,0.238324,2.447973,0.247536,0.01,300,"{'model__learning_rate': 0.01, 'model__max_ite...",0.750570,0.761069,0.756208,...,-0.042384,0.000668,5,-0.041128,-0.041476,-0.040868,-0.041080,-0.041235,-0.041157,0.000199
2,16.675906,0.233769,4.280070,0.700973,0.01,500,"{'model__learning_rate': 0.01, 'model__max_ite...",0.754703,0.768941,0.759522,...,-0.041088,0.000752,1,-0.039192,-0.039572,-0.038967,-0.039156,-0.039320,-0.039242,0.000200
3,2.646472,0.499324,0.831014,0.225845,0.10,100,"{'model__learning_rate': 0.1, 'model__max_iter...",0.755823,0.766906,0.761519,...,-0.041193,0.000842,2,-0.038506,-0.038840,-0.037865,-0.038097,-0.038172,-0.038296,0.000341
4,3.085408,0.372059,0.762590,0.094404,0.10,300,"{'model__learning_rate': 0.1, 'model__max_iter...",0.755823,0.766906,0.761519,...,-0.041193,0.000842,2,-0.038506,-0.038840,-0.037865,-0.038097,-0.038172,-0.038296,0.000341
5,2.705461,0.262062,0.720653,0.056310,0.10,500,"{'model__learning_rate': 0.1, 'model__max_iter...",0.755823,0.766906,0.761519,...,-0.041193,0.000842,2,-0.038506,-0.038840,-0.037865,-0.038097,-0.038172,-0.038296,0.000341
6,0.852980,0.197931,0.244238,0.041277,0.30,100,"{'model__learning_rate': 0.3, 'model__max_iter...",0.735446,0.762080,0.753824,...,-0.061612,0.014127,7,-0.088815,-0.044737,-0.053779,-0.043946,-0.057650,-0.057786,0.016375
7,0.900628,0.195587,0.256525,0.037344,0.30,300,"{'model__learning_rate': 0.3, 'model__max_iter...",0.735446,0.762080,0.753824,...,-0.061612,0.014127,7,-0.088815,-0.044737,-0.053779,-0.043946,-0.057650,-0.057786,0.016375
8,0.921457,0.201875,0.264277,0.071554,0.30,500,"{'model__learning_rate': 0.3, 'model__max_iter...",0.735446,0.762080,0.753824,...,-0.061612,0.014127,7,-0.088815,-0.044737,-0.053779,-0.043946,-0.057650,-0.057786,0.016375
9,0.601769,0.060869,0.161355,0.015134,0.50,100,"{'model__learning_rate': 0.5, 'model__max_iter...",0.657905,0.707692,0.730461,...,-0.294951,0.064037,10,-0.368625,-0.316588,-0.204586,-0.344959,-0.237022,-0.294356,0.063114


In [11]:
results_hgbc_gs2 = pd.DataFrame(grid_hgbc_gs2.cv_results_)
display(results_hgbc_gs2.head(10))

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__l2_regularization,param_model__max_depth,param_model__min_samples_leaf,params,split0_test_f1,split1_test_f1,...,mean_test_log_loss,std_test_log_loss,rank_test_log_loss,split0_train_log_loss,split1_train_log_loss,split2_train_log_loss,split3_train_log_loss,split4_train_log_loss,mean_train_log_loss,std_train_log_loss
0,2.391798,0.064450,0.430483,0.083173,0.0,1,1,"{'model__l2_regularization': 0.0, 'model__max_...",0.744398,0.746656,...,-0.043433,0.000699,31,-0.043321,-0.043624,-0.043139,-0.043226,-0.043364,-0.043335,0.000164
1,2.277947,0.214826,0.396235,0.035738,0.0,1,5,"{'model__l2_regularization': 0.0, 'model__max_...",0.744398,0.746656,...,-0.043433,0.000699,31,-0.043321,-0.043624,-0.043139,-0.043226,-0.043364,-0.043335,0.000164
2,2.090602,0.038460,0.392452,0.068325,0.0,1,10,"{'model__l2_regularization': 0.0, 'model__max_...",0.744398,0.746656,...,-0.043433,0.000699,31,-0.043321,-0.043624,-0.043139,-0.043226,-0.043364,-0.043335,0.000164
3,2.691709,0.078406,0.542287,0.051420,0.0,3,1,"{'model__l2_regularization': 0.0, 'model__max_...",0.754234,0.761544,...,-0.040831,0.000770,12,-0.040209,-0.040570,-0.039925,-0.040169,-0.040296,-0.040234,0.000208
4,2.679461,0.059369,0.540353,0.038068,0.0,3,5,"{'model__l2_regularization': 0.0, 'model__max_...",0.754234,0.761544,...,-0.040820,0.000760,6,-0.040200,-0.040570,-0.039942,-0.040177,-0.040320,-0.040242,0.000205
5,2.649600,0.072947,0.517783,0.053986,0.0,3,10,"{'model__l2_regularization': 0.0, 'model__max_...",0.753296,0.760968,...,-0.040819,0.000767,5,-0.040212,-0.040576,-0.039929,-0.040168,-0.040286,-0.040234,0.000208
6,3.102396,0.230613,0.651233,0.085505,0.0,5,1,"{'model__l2_regularization': 0.0, 'model__max_...",0.759390,0.775112,...,-0.040925,0.000764,18,-0.039404,-0.039774,-0.039043,-0.039214,-0.039756,-0.039438,0.000290
7,3.151025,0.224040,0.657448,0.095584,0.0,5,5,"{'model__l2_regularization': 0.0, 'model__max_...",0.760417,0.774242,...,-0.040862,0.000772,15,-0.039456,-0.039702,-0.038958,-0.039201,-0.039714,-0.039406,0.000292
8,3.375226,0.333733,0.853730,0.171045,0.0,5,10,"{'model__l2_regularization': 0.0, 'model__max_...",0.758467,0.774242,...,-0.040861,0.000802,14,-0.039437,-0.039803,-0.039051,-0.039234,-0.039668,-0.039439,0.000275
9,3.292233,0.234859,0.883772,0.144271,0.0,7,1,"{'model__l2_regularization': 0.0, 'model__max_...",0.756271,0.768127,...,-0.041203,0.000837,27,-0.038793,-0.039286,-0.038212,-0.038706,-0.038821,-0.038764,0.000342


In [12]:
results_hgbc_gs3 = pd.DataFrame(grid_hgbc_gs3.cv_results_)
display(results_hgbc_gs3.head(10))

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__l2_regularization,param_model__learning_rate,param_model__max_depth,param_model__max_iter,param_model__min_samples_leaf,params,...,mean_test_log_loss,std_test_log_loss,rank_test_log_loss,split0_train_log_loss,split1_train_log_loss,split2_train_log_loss,split3_train_log_loss,split4_train_log_loss,mean_train_log_loss,std_train_log_loss
0,2.205709,0.043389,0.349938,0.020585,0.0,0.01,1,100,1,"{'model__l2_regularization': 0.0, 'model__lear...",...,-0.069656,0.000362,316,-0.069511,-0.069887,-0.069470,-0.069489,-0.069835,-0.069638,0.000183
1,2.087458,0.107622,0.415125,0.068776,0.0,0.01,1,100,5,"{'model__l2_regularization': 0.0, 'model__lear...",...,-0.069656,0.000362,316,-0.069511,-0.069887,-0.069470,-0.069489,-0.069835,-0.069638,0.000183
2,2.014069,0.081126,0.474209,0.046815,0.0,0.01,1,100,10,"{'model__l2_regularization': 0.0, 'model__lear...",...,-0.069656,0.000362,316,-0.069511,-0.069887,-0.069470,-0.069489,-0.069835,-0.069638,0.000183
3,5.674650,0.064823,0.935589,0.054923,0.0,0.01,1,300,1,"{'model__l2_regularization': 0.0, 'model__lear...",...,-0.053033,0.000594,268,-0.052851,-0.053291,-0.052782,-0.052850,-0.053205,-0.052996,0.000209
4,5.760513,0.200603,0.954374,0.109215,0.0,0.01,1,300,5,"{'model__l2_regularization': 0.0, 'model__lear...",...,-0.053033,0.000594,268,-0.052851,-0.053291,-0.052782,-0.052850,-0.053205,-0.052996,0.000209
5,5.521683,0.042667,0.852291,0.016709,0.0,0.01,1,300,10,"{'model__l2_regularization': 0.0, 'model__lear...",...,-0.053033,0.000594,268,-0.052851,-0.053291,-0.052782,-0.052850,-0.053205,-0.052996,0.000209
6,9.236172,0.238199,1.350607,0.077399,0.0,0.01,1,500,1,"{'model__l2_regularization': 0.0, 'model__lear...",...,-0.047560,0.000625,241,-0.047398,-0.047791,-0.047289,-0.047345,-0.047656,-0.047496,0.000194
7,9.260495,0.384143,1.434025,0.083934,0.0,0.01,1,500,5,"{'model__l2_regularization': 0.0, 'model__lear...",...,-0.047560,0.000625,241,-0.047398,-0.047791,-0.047289,-0.047345,-0.047656,-0.047496,0.000194
8,9.422484,0.095444,1.418532,0.052300,0.0,0.01,1,500,10,"{'model__l2_regularization': 0.0, 'model__lear...",...,-0.047560,0.000625,241,-0.047398,-0.047791,-0.047289,-0.047345,-0.047656,-0.047496,0.000194
9,2.572685,0.198901,0.517895,0.035846,0.0,0.01,3,100,1,"{'model__l2_regularization': 0.0, 'model__lear...",...,-0.059467,0.000465,298,-0.059285,-0.059574,-0.059207,-0.059241,-0.059512,-0.059364,0.000150


In [13]:
def summarize_grid_hgbc(grid, model_name):
    results = pd.DataFrame(grid.cv_results_)
    best = results.loc[grid.best_index_]

    print(f"\n===== {model_name} =====")
    print(f"Meilleurs paramètres : {grid.best_params_}")
    print(f"F1 moyen       : {best['mean_test_f1']:.4f}")
    print(f"PR-AUC moyen   : {best['mean_test_pr_auc']:.4f}")
    print(f"Recall moyen   : {best['mean_test_recall']:.4f}")
    print(f"Log-loss moyen : {-best['mean_test_log_loss']:.4f}")

    return results.sort_values("rank_test_pr_auc")


results_hgbc_gs1 = summarize_grid_hgbc(grid_hgbc_gs1, "Gradient Boosting Classifier - Grid Search 1")
results_hgbc_gs2 = summarize_grid_hgbc(grid_hgbc_gs2, "Gradient Boosting Classifier - Grid Search 2")
results_hgbc_gs3 = summarize_grid_hgbc(grid_hgbc_gs3, "Gradient Boosting Classifier - Grid Search 3")

display(results_hgbc_gs1.head(10))
display(results_hgbc_gs2.head(10))
display(results_hgbc_gs3.head(10))


===== Gradient Boosting Classifier - Grid Search 1 =====
Meilleurs paramètres : {'model__learning_rate': 0.01, 'model__max_iter': 500}
F1 moyen       : 0.7611
PR-AUC moyen   : 0.8420
Recall moyen   : 0.6912
Log-loss moyen : 0.0411

===== Gradient Boosting Classifier - Grid Search 2 =====
Meilleurs paramètres : {'model__l2_regularization': 1.0, 'model__max_depth': 5, 'model__min_samples_leaf': 1}
F1 moyen       : 0.7655
PR-AUC moyen   : 0.8428
Recall moyen   : 0.7062
Log-loss moyen : 0.0408

===== Gradient Boosting Classifier - Grid Search 3 =====
Meilleurs paramètres : {'model__l2_regularization': 1.0, 'model__learning_rate': 0.1, 'model__max_depth': 5, 'model__max_iter': 300, 'model__min_samples_leaf': 1}
F1 moyen       : 0.7657
PR-AUC moyen   : 0.8428
Recall moyen   : 0.7063
Log-loss moyen : 0.0408


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__learning_rate,param_model__max_iter,params,split0_test_f1,split1_test_f1,split2_test_f1,...,mean_test_log_loss,std_test_log_loss,rank_test_log_loss,split0_train_log_loss,split1_train_log_loss,split2_train_log_loss,split3_train_log_loss,split4_train_log_loss,mean_train_log_loss,std_train_log_loss
2,16.675906,0.233769,4.280070,0.700973,0.01,500,"{'model__learning_rate': 0.01, 'model__max_ite...",0.754703,0.768941,0.759522,...,-0.041088,0.000752,1,-0.039192,-0.039572,-0.038967,-0.039156,-0.039320,-0.039242,0.000200
1,9.599144,0.238324,2.447973,0.247536,0.01,300,"{'model__learning_rate': 0.01, 'model__max_ite...",0.750570,0.761069,0.756208,...,-0.042384,0.000668,5,-0.041128,-0.041476,-0.040868,-0.041080,-0.041235,-0.041157,0.000199
3,2.646472,0.499324,0.831014,0.225845,0.10,100,"{'model__learning_rate': 0.1, 'model__max_iter...",0.755823,0.766906,0.761519,...,-0.041193,0.000842,2,-0.038506,-0.038840,-0.037865,-0.038097,-0.038172,-0.038296,0.000341
4,3.085408,0.372059,0.762590,0.094404,0.10,300,"{'model__learning_rate': 0.1, 'model__max_iter...",0.755823,0.766906,0.761519,...,-0.041193,0.000842,2,-0.038506,-0.038840,-0.037865,-0.038097,-0.038172,-0.038296,0.000341
5,2.705461,0.262062,0.720653,0.056310,0.10,500,"{'model__learning_rate': 0.1, 'model__max_iter...",0.755823,0.766906,0.761519,...,-0.041193,0.000842,2,-0.038506,-0.038840,-0.037865,-0.038097,-0.038172,-0.038296,0.000341
0,3.727680,0.081667,0.677894,0.029321,0.01,100,"{'model__learning_rate': 0.01, 'model__max_ite...",0.695764,0.694600,0.695283,...,-0.056851,0.000408,6,-0.056222,-0.056448,-0.055999,-0.056169,-0.056307,-0.056229,0.000149
6,0.852980,0.197931,0.244238,0.041277,0.30,100,"{'model__learning_rate': 0.3, 'model__max_iter...",0.735446,0.762080,0.753824,...,-0.061612,0.014127,7,-0.088815,-0.044737,-0.053779,-0.043946,-0.057650,-0.057786,0.016375
7,0.900628,0.195587,0.256525,0.037344,0.30,300,"{'model__learning_rate': 0.3, 'model__max_iter...",0.735446,0.762080,0.753824,...,-0.061612,0.014127,7,-0.088815,-0.044737,-0.053779,-0.043946,-0.057650,-0.057786,0.016375
8,0.921457,0.201875,0.264277,0.071554,0.30,500,"{'model__learning_rate': 0.3, 'model__max_iter...",0.735446,0.762080,0.753824,...,-0.061612,0.014127,7,-0.088815,-0.044737,-0.053779,-0.043946,-0.057650,-0.057786,0.016375
9,0.601769,0.060869,0.161355,0.015134,0.50,100,"{'model__learning_rate': 0.5, 'model__max_iter...",0.657905,0.707692,0.730461,...,-0.294951,0.064037,10,-0.368625,-0.316588,-0.204586,-0.344959,-0.237022,-0.294356,0.063114


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__l2_regularization,param_model__max_depth,param_model__min_samples_leaf,params,split0_test_f1,split1_test_f1,...,mean_test_log_loss,std_test_log_loss,rank_test_log_loss,split0_train_log_loss,split1_train_log_loss,split2_train_log_loss,split3_train_log_loss,split4_train_log_loss,mean_train_log_loss,std_train_log_loss
32,3.069362,0.161916,0.787911,0.065725,1.0,5,10,"{'model__l2_regularization': 1.0, 'model__max_...",0.757722,0.773084,...,-0.040805,0.000742,1,-0.039450,-0.039975,-0.039199,-0.039431,-0.039732,-0.039557,0.000269
17,2.687146,0.057255,0.510921,0.011908,0.5,3,10,"{'model__l2_regularization': 0.5, 'model__max_...",0.753198,0.761437,...,-0.040821,0.000749,9,-0.040234,-0.040602,-0.039930,-0.040176,-0.040316,-0.040252,0.000218
29,2.750636,0.071494,0.550089,0.045315,1.0,3,10,"{'model__l2_regularization': 1.0, 'model__max_...",0.754049,0.760030,...,-0.040821,0.000762,8,-0.040205,-0.040617,-0.039971,-0.040197,-0.040311,-0.040260,0.000210
30,3.133334,0.233959,0.779550,0.088061,1.0,5,1,"{'model__l2_regularization': 1.0, 'model__max_...",0.759259,0.773323,...,-0.040818,0.000726,4,-0.039529,-0.039915,-0.039168,-0.039347,-0.039849,-0.039561,0.000286
16,2.799482,0.128333,0.569826,0.067284,0.5,3,5,"{'model__l2_regularization': 0.5, 'model__max_...",0.754333,0.761617,...,-0.040820,0.000752,7,-0.040204,-0.040598,-0.039935,-0.040181,-0.040334,-0.040250,0.000217
28,2.774654,0.064085,0.537849,0.027557,1.0,3,5,"{'model__l2_regularization': 1.0, 'model__max_...",0.754902,0.759561,...,-0.040815,0.000763,2,-0.040224,-0.040605,-0.039971,-0.040198,-0.040331,-0.040266,0.000206
27,2.931345,0.052298,0.547639,0.052298,1.0,3,1,"{'model__l2_regularization': 1.0, 'model__max_...",0.754902,0.760030,...,-0.040818,0.000763,3,-0.040222,-0.040608,-0.039953,-0.040200,-0.040339,-0.040265,0.000213
20,2.934715,0.239930,0.704057,0.093251,0.5,5,10,"{'model__l2_regularization': 0.5, 'model__max_...",0.757440,0.774506,...,-0.040846,0.000770,13,-0.039495,-0.039966,-0.039225,-0.039382,-0.039812,-0.039576,0.000274
31,2.999884,0.153481,0.707809,0.028977,1.0,5,5,"{'model__l2_regularization': 1.0, 'model__max_...",0.759362,0.773783,...,-0.040822,0.000725,10,-0.039511,-0.039946,-0.039246,-0.039424,-0.039746,-0.039575,0.000246
15,2.741390,0.063920,0.540751,0.007531,0.5,3,1,"{'model__l2_regularization': 0.5, 'model__max_...",0.754333,0.762552,...,-0.040826,0.000745,11,-0.040204,-0.040606,-0.039952,-0.040170,-0.040327,-0.040252,0.000214


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__l2_regularization,param_model__learning_rate,param_model__max_depth,param_model__max_iter,param_model__min_samples_leaf,params,...,mean_test_log_loss,std_test_log_loss,rank_test_log_loss,split0_train_log_loss,split1_train_log_loss,split2_train_log_loss,split3_train_log_loss,split4_train_log_loss,mean_train_log_loss,std_train_log_loss
365,5.267026,0.889078,0.831367,0.121697,1.0,0.3,1,300,10,"{'model__l2_regularization': 1.0, 'model__lear...",...,-0.040599,0.000713,4,-0.040387,-0.040618,-0.040051,-0.040236,-0.040511,-0.040361,0.000201
364,5.199252,0.993735,0.797368,0.084488,1.0,0.3,1,300,5,"{'model__l2_regularization': 1.0, 'model__lear...",...,-0.040599,0.000713,4,-0.040387,-0.040618,-0.040051,-0.040236,-0.040511,-0.040361,0.000201
363,5.016602,0.928492,0.720869,0.140068,1.0,0.3,1,300,1,"{'model__l2_regularization': 1.0, 'model__lear...",...,-0.040599,0.000713,4,-0.040387,-0.040618,-0.040051,-0.040236,-0.040511,-0.040361,0.000201
367,5.690377,1.478828,0.867073,0.190484,1.0,0.3,1,500,5,"{'model__l2_regularization': 1.0, 'model__lear...",...,-0.040602,0.000718,10,-0.040387,-0.040618,-0.040036,-0.040218,-0.040511,-0.040354,0.000207
366,5.652598,1.418519,0.864487,0.218268,1.0,0.3,1,500,1,"{'model__l2_regularization': 1.0, 'model__lear...",...,-0.040602,0.000718,10,-0.040387,-0.040618,-0.040036,-0.040218,-0.040511,-0.040354,0.000207
368,5.863390,1.510607,0.914145,0.163752,1.0,0.3,1,500,10,"{'model__l2_regularization': 1.0, 'model__lear...",...,-0.040602,0.000718,10,-0.040387,-0.040618,-0.040036,-0.040218,-0.040511,-0.040354,0.000207
221,5.250437,0.804679,0.827513,0.166444,0.5,0.3,1,300,10,"{'model__l2_regularization': 0.5, 'model__lear...",...,-0.040598,0.000708,1,-0.040378,-0.040626,-0.040052,-0.040233,-0.040501,-0.040358,0.000201
220,5.521467,0.896676,0.886735,0.214326,0.5,0.3,1,300,5,"{'model__l2_regularization': 0.5, 'model__lear...",...,-0.040598,0.000708,1,-0.040378,-0.040626,-0.040052,-0.040233,-0.040501,-0.040358,0.000201
219,5.506170,0.940333,0.858330,0.256520,0.5,0.3,1,300,1,"{'model__l2_regularization': 0.5, 'model__lear...",...,-0.040598,0.000708,1,-0.040378,-0.040626,-0.040052,-0.040233,-0.040501,-0.040358,0.000201
224,6.228595,2.440539,0.911778,0.267267,0.5,0.3,1,500,10,"{'model__l2_regularization': 0.5, 'model__lear...",...,-0.040600,0.000709,7,-0.040378,-0.040626,-0.040052,-0.040197,-0.040501,-0.040351,0.000206


In [14]:
model_scores = pd.DataFrame({
    "model": ["HGBG GS1", "HGBG GS2", "HGBG GS3"],
    "mean_cv_f1": [
        grid_hgbc_gs1.best_score_,
        grid_hgbc_gs2.best_score_,
        grid_hgbc_gs3.best_score_,
    ],
})

display(model_scores.sort_values("mean_cv_f1", ascending=False))

grids = {
    "HGBG GS1": grid_hgbc_gs1,
    "HGBG GS2": grid_hgbc_gs2,
    "HGBG GS3": grid_hgbc_gs3,
}

best_model_name = max(
    grids,
    key=lambda name: grids[name].best_score_
)

best_grid = grids[best_model_name]
best_model_hgbc = best_grid.best_estimator_

print(f"Modèle sélectionné : {best_model_name}")
print(f"F1 moyen en validation croisée : {best_grid.best_score_:.4f}")
print(f"Meilleurs paramètres : {best_grid.best_params_}")
print(f"Meilleur modèle évalué: {best_model_hgbc}") 

,model,mean_cv_f1
2,HGBG GS3,0.765689
1,HGBG GS2,0.765541
0,HGBG GS1,0.761147


Modèle sélectionné : HGBG GS3
F1 moyen en validation croisée : 0.7657
Meilleurs paramètres : {'model__l2_regularization': 1.0, 'model__learning_rate': 0.1, 'model__max_depth': 5, 'model__max_iter': 300, 'model__min_samples_leaf': 1}
Meilleur modèle évalué: Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['age',
                                                   'total_pages_visited']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['country', 'source',
                                                   'new_user'])])),
                ('model',
                 HistGradientBoostingClassifier(l2_regularization=1.0,
                                   

In [15]:
# Optimisation du seuil
tuned_model = TunedThresholdClassifierCV(
    estimator=best_model_hgbc,
    scoring="f1",
    thresholds=100,
    cv=5,
    n_jobs=-1,
    store_cv_results=True
)

tuned_model.fit(X_train, y_train)

print("Seuil optimal :", tuned_model.best_threshold_)
print("F1 CV estimé :", tuned_model.best_score_)

Seuil optimal : 0.43423267545238436
F1 CV estimé : 0.7713812735569622


#### Les hyperparamètres retenus pour le modèle de HistGradientBoostingClassifier sont les suivants :
- learning_rate : 
- max_depth : 
- max_leaf_nodes :
- min_samples_leaf :
- max_iter :
- L2 regularization :
- Seuil de classification :

#### Modèle 2 : Logistic Regression

In [16]:
numeric_features = ["age", "total_pages_visited"]
categorical_features = ["country", "source", "new_user"] # new_user est à exclure car déjà un int64 ? 

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(drop="first")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [17]:
# Gridsearch régularisation elasticnet : 
params_lr_elasticnet = {
    "model__penalty": ["elasticnet"],
    "model__l1_ratio": [0.0, 0.5, 1.0],
    "model__max_iter": [300, 500, 1000],
    "model__C": [0.01, 0.1, 1.0, 10.0, 100.0],
    "model__solver": ["saga"],
    "model__class_weight": [None, "balanced"],
    "model__tol": [1e-4, 1e-3, 1e-2]
}

# Gridsearch régularisation Ridge :
params_lr_ridge = {
    "model__penalty": ["l2"],
    "model__max_iter": [300, 500, 1000],
    "model__C": [0.01, 0.1, 1.0, 10.0, 100.0],
    "model__solver": ["saga", "lbfgs", "liblinear"],
    "model__class_weight": [None, "balanced"],
    "model__tol": [1e-4, 1e-3, 1e-2]
    }

# Gridsearch régularisation Lasso :
params_lr_lasso = {
    "model__penalty": ["l1"],
    "model__max_iter": [300, 500, 1000],
    "model__C": [0.01, 0.1, 1.0, 10.0, 100.0],
    "model__solver": ["saga", "liblinear"],
    "model__class_weight": [None, "balanced"],
    "model__tol": [1e-4, 1e-3, 1e-2]
    }

In [18]:
model = LogisticRegression(random_state=42)

pipeline_lr = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model),
])

scoring = {
    "f1": "f1",
    "recall": "recall",
    "pr_auc": "average_precision",
}

In [19]:
grid_lr_lasso = GridSearchCV(
    estimator=pipeline_lr,
    param_grid=params_lr_lasso,
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)
grid_lr_lasso.fit(X_train, y_train)

grid_lr_ridge = GridSearchCV(
    estimator=pipeline_lr,
    param_grid=params_lr_ridge,
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)
grid_lr_ridge.fit(X_train, y_train)

grid_lr_elasticnet = GridSearchCV(
    estimator=pipeline_lr,
    param_grid=params_lr_elasticnet,
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)
grid_lr_elasticnet.fit(X_train, y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         StandardScaler(),
                                                                         ['age',
                                                                          'total_pages_visited']),
                                                                        ('cat',
                                                                         OneHotEncoder(drop='first'),
                                                                         ['country',
                                                                          'source',
                                                                          'new_user'])])),
                                       ('model',
                                        LogisticRegression(random_state=42))]),
             n_jobs=-1,
             param_grid={'model__C': [0.01, 0.1, 1.0, 10.0, 100.0],
                         'model__class_weight': [None, 'balanced'],
                         'model__l1_ratio': [0.0, 0.5, 1.0],
                         'model__max_iter': [300, 500, 1000],
                         'model__penalty': ['elasticnet'],
                         'model__solver': ['saga'],
                         'model__tol': [0.0001, 0.001, 0.01]},
             refit='f1', return_train_score=True,
             scoring={'f1': 'f1', 'pr_auc': 'average_precision',
                      'recall': 'recall'})

In [20]:
results_lr_lasso = pd.DataFrame(grid_lr_lasso.cv_results_)
display(results_lr_lasso.head(10))

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__C,param_model__class_weight,param_model__max_iter,param_model__penalty,param_model__solver,param_model__tol,...,mean_test_pr_auc,std_test_pr_auc,rank_test_pr_auc,split0_train_pr_auc,split1_train_pr_auc,split2_train_pr_auc,split3_train_pr_auc,split4_train_pr_auc,mean_train_pr_auc,std_train_pr_auc
0,1.351772,0.060992,0.114171,0.009986,0.01,None,300,l1,saga,0.0001,...,0.834808,0.005260,169,0.834689,0.833029,0.836410,0.836189,0.833741,0.834811,0.001326
1,1.007624,0.117621,0.105133,0.010273,0.01,None,300,l1,saga,0.0010,...,0.834814,0.005243,166,0.834686,0.833032,0.836460,0.836189,0.833725,0.834819,0.001340
2,0.797653,0.075594,0.118056,0.016430,0.01,None,300,l1,saga,0.0100,...,0.834920,0.005149,163,0.834840,0.833016,0.836726,0.836425,0.833747,0.834951,0.001451
3,1.013950,0.176648,0.160505,0.025137,0.01,None,300,l1,liblinear,0.0001,...,0.831595,0.005326,172,0.831544,0.829946,0.833337,0.832860,0.830377,0.831613,0.001330
4,0.968857,0.094122,0.141642,0.034313,0.01,None,300,l1,liblinear,0.0010,...,0.831477,0.005351,175,0.831463,0.829852,0.833263,0.832642,0.830227,0.831489,0.001323
5,0.530849,0.025495,0.119204,0.014474,0.01,None,300,l1,liblinear,0.0100,...,0.829702,0.004577,178,0.830112,0.826994,0.832120,0.831846,0.827345,0.829683,0.002168
6,1.420586,0.105327,0.112997,0.011031,0.01,None,500,l1,saga,0.0001,...,0.834808,0.005260,169,0.834689,0.833029,0.836410,0.836189,0.833741,0.834811,0.001326
7,0.928587,0.122463,0.117725,0.021106,0.01,None,500,l1,saga,0.0010,...,0.834814,0.005243,166,0.834686,0.833032,0.836460,0.836189,0.833725,0.834819,0.001340
8,0.773199,0.104158,0.119875,0.012455,0.01,None,500,l1,saga,0.0100,...,0.834920,0.005149,163,0.834840,0.833016,0.836726,0.836425,0.833747,0.834951,0.001451
9,0.859274,0.111635,0.131674,0.020183,0.01,None,500,l1,liblinear,0.0001,...,0.831595,0.005326,172,0.831544,0.829946,0.833337,0.832860,0.830377,0.831613,0.001330


In [21]:
results_lr_ridge = pd.DataFrame(grid_lr_ridge.cv_results_)
display(results_lr_ridge.head(10))

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__C,param_model__class_weight,param_model__max_iter,param_model__penalty,param_model__solver,param_model__tol,...,mean_test_pr_auc,std_test_pr_auc,rank_test_pr_auc,split0_train_pr_auc,split1_train_pr_auc,split2_train_pr_auc,split3_train_pr_auc,split4_train_pr_auc,mean_train_pr_auc,std_train_pr_auc
0,1.181162,0.127849,0.108623,0.011182,0.01,None,300,l2,saga,0.0001,...,0.837114,0.005467,223,0.837199,0.835415,0.838887,0.838426,0.836074,0.837200,0.001327
1,0.871308,0.048032,0.101368,0.013255,0.01,None,300,l2,saga,0.0010,...,0.837116,0.005467,220,0.837189,0.835407,0.838888,0.838428,0.836088,0.837200,0.001327
2,0.625917,0.066110,0.098114,0.005825,0.01,None,300,l2,saga,0.0100,...,0.837195,0.005465,217,0.837303,0.835418,0.838932,0.838539,0.836113,0.837261,0.001353
3,0.380085,0.044633,0.113048,0.016568,0.01,None,300,l2,lbfgs,0.0001,...,0.837095,0.005435,226,0.837218,0.835398,0.838901,0.838447,0.835964,0.837185,0.001358
4,0.333719,0.033869,0.107840,0.004950,0.01,None,300,l2,lbfgs,0.0010,...,0.836039,0.005409,229,0.836043,0.834278,0.837742,0.837309,0.835066,0.836088,0.001308
5,0.237928,0.027862,0.120625,0.011114,0.01,None,300,l2,lbfgs,0.0100,...,0.802846,0.006424,256,0.803487,0.801294,0.804187,0.804358,0.800799,0.802825,0.001490
6,0.537847,0.034360,0.161235,0.037552,0.01,None,300,l2,liblinear,0.0001,...,0.827509,0.005976,232,0.827786,0.826003,0.829143,0.828982,0.826155,0.827614,0.001339
7,0.592230,0.074991,0.171217,0.036565,0.01,None,300,l2,liblinear,0.0010,...,0.827497,0.005968,235,0.827771,0.825982,0.829145,0.828982,0.826126,0.827601,0.001350
8,0.500762,0.027472,0.113136,0.009030,0.01,None,300,l2,liblinear,0.0100,...,0.827475,0.005992,238,0.827771,0.825982,0.829145,0.828879,0.826126,0.827581,0.001330
9,1.137824,0.116316,0.111567,0.021531,0.01,None,500,l2,saga,0.0001,...,0.837114,0.005467,223,0.837199,0.835415,0.838887,0.838426,0.836074,0.837200,0.001327


In [22]:
results_lr_elasticnet = pd.DataFrame(grid_lr_elasticnet.cv_results_)
display(results_lr_elasticnet.head(10))

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__C,param_model__class_weight,param_model__l1_ratio,param_model__max_iter,param_model__penalty,param_model__solver,...,mean_test_pr_auc,std_test_pr_auc,rank_test_pr_auc,split0_train_pr_auc,split1_train_pr_auc,split2_train_pr_auc,split3_train_pr_auc,split4_train_pr_auc,mean_train_pr_auc,std_train_pr_auc
0,1.168629,0.129387,0.131790,0.018184,0.01,None,0.0,300,elasticnet,saga,...,0.837114,0.005467,250,0.837199,0.835415,0.838887,0.838426,0.836074,0.837200,0.001327
1,0.852246,0.082180,0.124137,0.021474,0.01,None,0.0,300,elasticnet,saga,...,0.837116,0.005467,247,0.837189,0.835407,0.838888,0.838428,0.836088,0.837200,0.001327
2,0.607759,0.080794,0.116556,0.023619,0.01,None,0.0,300,elasticnet,saga,...,0.837195,0.005465,244,0.837303,0.835418,0.838932,0.838539,0.836113,0.837261,0.001353
3,1.076599,0.063024,0.100382,0.006744,0.01,None,0.0,500,elasticnet,saga,...,0.837114,0.005467,250,0.837199,0.835415,0.838887,0.838426,0.836074,0.837200,0.001327
4,0.840567,0.051739,0.098865,0.011894,0.01,None,0.0,500,elasticnet,saga,...,0.837116,0.005467,247,0.837189,0.835407,0.838888,0.838428,0.836088,0.837200,0.001327
5,0.609812,0.089069,0.099987,0.005878,0.01,None,0.0,500,elasticnet,saga,...,0.837195,0.005465,244,0.837303,0.835418,0.838932,0.838539,0.836113,0.837261,0.001353
6,1.154237,0.199558,0.102201,0.007776,0.01,None,0.0,1000,elasticnet,saga,...,0.837114,0.005467,250,0.837199,0.835415,0.838887,0.838426,0.836074,0.837200,0.001327
7,0.782757,0.045025,0.099977,0.014838,0.01,None,0.0,1000,elasticnet,saga,...,0.837116,0.005467,247,0.837189,0.835407,0.838888,0.838428,0.836088,0.837200,0.001327
8,0.674449,0.077490,0.104200,0.014726,0.01,None,0.0,1000,elasticnet,saga,...,0.837195,0.005465,244,0.837303,0.835418,0.838932,0.838539,0.836113,0.837261,0.001353
9,1.239131,0.126180,0.109832,0.008122,0.01,None,0.5,300,elasticnet,saga,...,0.836475,0.005351,256,0.836553,0.834728,0.838233,0.837775,0.835346,0.836527,0.001349


In [23]:
def summarize_grid_lr(grid, model_name):
    results = pd.DataFrame(grid.cv_results_)
    best = results.loc[grid.best_index_]

    print(f"\n===== {model_name} =====")
    print(f"Meilleurs paramètres : {grid.best_params_}")
    print(f"F1 moyen       : {best['mean_test_f1']:.4f}")
    print(f"PR-AUC moyen   : {best['mean_test_pr_auc']:.4f}")
    print(f"Recall moyen   : {best['mean_test_recall']:.4f}")

    return results.sort_values("rank_test_pr_auc")


results_lr_lasso = summarize_grid_lr(grid_lr_lasso, "Logistic Regression Lasso - Grid Search")
results_lr_ridge = summarize_grid_lr(grid_lr_ridge, "Logistic Regression Ridge - Grid Search")
results_lr_elasticnet = summarize_grid_lr(grid_lr_elasticnet, "Logistic Regression ElasticNet - Grid Search")

display(results_lr_lasso.head(10))
display(results_lr_ridge.head(10))
display(results_lr_elasticnet.head(10))


===== Logistic Regression Lasso - Grid Search =====
Meilleurs paramètres : {'model__C': 10.0, 'model__class_weight': None, 'model__max_iter': 300, 'model__penalty': 'l1', 'model__solver': 'saga', 'model__tol': 0.0001}
F1 moyen       : 0.7641
PR-AUC moyen   : 0.8452
Recall moyen   : 0.6919

===== Logistic Regression Ridge - Grid Search =====
Meilleurs paramètres : {'model__C': 100.0, 'model__class_weight': None, 'model__max_iter': 300, 'model__penalty': 'l2', 'model__solver': 'liblinear', 'model__tol': 0.01}
F1 moyen       : 0.7645
PR-AUC moyen   : 0.8452
Recall moyen   : 0.6926

===== Logistic Regression ElasticNet - Grid Search =====
Meilleurs paramètres : {'model__C': 10.0, 'model__class_weight': None, 'model__l1_ratio': 0.5, 'model__max_iter': 300, 'model__penalty': 'elasticnet', 'model__solver': 'saga', 'model__tol': 0.0001}
F1 moyen       : 0.7641
PR-AUC moyen   : 0.8452
Recall moyen   : 0.6919


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__C,param_model__class_weight,param_model__max_iter,param_model__penalty,param_model__solver,param_model__tol,...,mean_test_pr_auc,std_test_pr_auc,rank_test_pr_auc,split0_train_pr_auc,split1_train_pr_auc,split2_train_pr_auc,split3_train_pr_auc,split4_train_pr_auc,mean_train_pr_auc,std_train_pr_auc
86,0.743467,0.091297,0.104443,0.003649,1.0,None,1000,l1,saga,0.0100,...,0.845237,0.005676,1,0.845412,0.843229,0.846987,0.846663,0.844502,0.845358,0.001387
74,0.861035,0.136899,0.119881,0.023065,1.0,None,300,l1,saga,0.0100,...,0.845237,0.005676,1,0.845412,0.843229,0.846987,0.846663,0.844502,0.845358,0.001387
80,0.736985,0.112981,0.110767,0.006346,1.0,None,500,l1,saga,0.0100,...,0.845237,0.005676,1,0.845412,0.843229,0.846987,0.846663,0.844502,0.845358,0.001387
148,2.100253,0.284656,0.136207,0.005365,100.0,None,300,l1,liblinear,0.0010,...,0.845237,0.005690,4,0.845417,0.843228,0.846981,0.846668,0.844503,0.845359,0.001387
154,2.261693,0.355564,0.132214,0.015709,100.0,None,500,l1,liblinear,0.0010,...,0.845237,0.005690,4,0.845417,0.843228,0.846981,0.846668,0.844503,0.845359,0.001387
160,2.195437,0.291027,0.153807,0.016181,100.0,None,1000,l1,liblinear,0.0010,...,0.845237,0.005690,4,0.845417,0.843228,0.846981,0.846668,0.844503,0.845359,0.001387
87,2.870435,0.204186,0.150840,0.014219,1.0,None,1000,l1,liblinear,0.0001,...,0.845236,0.005679,7,0.845411,0.843230,0.846973,0.846666,0.844501,0.845356,0.001385
81,2.977202,0.145149,0.162789,0.009888,1.0,None,500,l1,liblinear,0.0001,...,0.845236,0.005679,7,0.845411,0.843230,0.846973,0.846666,0.844501,0.845356,0.001385
75,2.966911,0.151885,0.180295,0.047538,1.0,None,300,l1,liblinear,0.0001,...,0.845236,0.005679,7,0.845411,0.843230,0.846973,0.846666,0.844501,0.845356,0.001385
109,1.164108,0.080044,0.121790,0.020062,10.0,None,300,l1,saga,0.0010,...,0.845236,0.005685,10,0.845415,0.843224,0.846981,0.846669,0.844507,0.845359,0.001388


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__C,param_model__class_weight,param_model__max_iter,param_model__penalty,param_model__solver,param_model__tol,...,mean_test_pr_auc,std_test_pr_auc,rank_test_pr_auc,split0_train_pr_auc,split1_train_pr_auc,split2_train_pr_auc,split3_train_pr_auc,split4_train_pr_auc,mean_train_pr_auc,std_train_pr_auc
241,0.660277,0.070358,0.115973,0.007896,100.0,None,1000,l2,liblinear,0.001,...,0.845243,0.005689,1,0.845420,0.843224,0.846983,0.846669,0.844508,0.845361,0.001388
232,0.638723,0.028355,0.133226,0.015528,100.0,None,500,l2,liblinear,0.001,...,0.845243,0.005689,1,0.845420,0.843224,0.846983,0.846669,0.844508,0.845361,0.001388
223,0.687257,0.078727,0.112711,0.010899,100.0,None,300,l2,liblinear,0.001,...,0.845243,0.005689,1,0.845420,0.843224,0.846983,0.846669,0.844508,0.845361,0.001388
233,0.688082,0.066136,0.158701,0.031317,100.0,None,500,l2,liblinear,0.010,...,0.845243,0.005689,4,0.845403,0.843227,0.846975,0.846664,0.844482,0.845350,0.001388
224,0.592532,0.028435,0.120076,0.019993,100.0,None,300,l2,liblinear,0.010,...,0.845243,0.005689,4,0.845403,0.843227,0.846975,0.846664,0.844482,0.845350,0.001388
242,0.561105,0.031626,0.158251,0.032108,100.0,None,1000,l2,liblinear,0.010,...,0.845243,0.005689,4,0.845403,0.843227,0.846975,0.846664,0.844482,0.845350,0.001388
169,0.671941,0.036525,0.154007,0.017355,10.0,None,300,l2,liblinear,0.001,...,0.845239,0.005686,7,0.845416,0.843223,0.846983,0.846665,0.844507,0.845359,0.001388
187,0.648243,0.019616,0.127287,0.012651,10.0,None,1000,l2,liblinear,0.001,...,0.845239,0.005686,7,0.845416,0.843223,0.846983,0.846665,0.844507,0.845359,0.001388
178,0.705205,0.053542,0.128742,0.011960,10.0,None,500,l2,liblinear,0.001,...,0.845239,0.005686,7,0.845416,0.843223,0.846983,0.846665,0.844507,0.845359,0.001388
226,1.016177,0.106642,0.099713,0.003271,100.0,None,500,l2,saga,0.001,...,0.845236,0.005684,10,0.845418,0.843225,0.846982,0.846665,0.844506,0.845359,0.001387


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__C,param_model__class_weight,param_model__l1_ratio,param_model__max_iter,param_model__penalty,param_model__solver,...,mean_test_pr_auc,std_test_pr_auc,rank_test_pr_auc,split0_train_pr_auc,split1_train_pr_auc,split2_train_pr_auc,split3_train_pr_auc,split4_train_pr_auc,mean_train_pr_auc,std_train_pr_auc
134,0.834057,0.095146,0.096975,0.010564,1.0,None,1.0,1000,elasticnet,saga,...,0.845237,0.005676,1,0.845412,0.843229,0.846987,0.846663,0.844502,0.845358,0.001387
128,0.761342,0.036197,0.099223,0.011653,1.0,None,1.0,300,elasticnet,saga,...,0.845237,0.005676,1,0.845412,0.843229,0.846987,0.846663,0.844502,0.845358,0.001387
131,0.727632,0.070916,0.100567,0.015408,1.0,None,1.0,500,elasticnet,saga,...,0.845237,0.005676,1,0.845412,0.843229,0.846987,0.846663,0.844502,0.845358,0.001387
177,1.687496,0.127706,0.098273,0.029286,10.0,None,0.5,1000,elasticnet,saga,...,0.845237,0.005684,4,0.845415,0.843223,0.846984,0.846668,0.844509,0.845360,0.001389
174,1.725623,0.124358,0.116056,0.007399,10.0,None,0.5,500,elasticnet,saga,...,0.845237,0.005684,4,0.845415,0.843223,0.846984,0.846668,0.844509,0.845360,0.001389
171,1.856949,0.110774,0.109422,0.010841,10.0,None,0.5,300,elasticnet,saga,...,0.845237,0.005684,4,0.845415,0.843223,0.846984,0.846668,0.844509,0.845360,0.001389
175,1.177360,0.074818,0.105252,0.012608,10.0,None,0.5,500,elasticnet,saga,...,0.845236,0.005681,7,0.845415,0.843225,0.846985,0.846665,0.844506,0.845359,0.001388
172,1.221264,0.050401,0.119935,0.036502,10.0,None,0.5,300,elasticnet,saga,...,0.845236,0.005681,7,0.845415,0.843225,0.846985,0.846665,0.844506,0.845359,0.001388
178,1.185875,0.104634,0.108476,0.015052,10.0,None,0.5,1000,elasticnet,saga,...,0.845236,0.005681,7,0.845415,0.843225,0.846985,0.846665,0.844506,0.845359,0.001388
181,1.138561,0.120992,0.101185,0.008951,10.0,None,1.0,300,elasticnet,saga,...,0.845236,0.005685,10,0.845415,0.843224,0.846981,0.846669,0.844507,0.845359,0.001388


In [24]:
model_scores = pd.DataFrame({
    "model": ["Logitic Regression - Lasso", "Logitic Regression - Ridge", "Logitic Regression - ElasticNet"],
    "mean_cv_f1": [
        grid_lr_lasso.best_score_,
        grid_lr_ridge.best_score_,
        grid_lr_elasticnet.best_score_,
    ],
})

display(model_scores.sort_values("mean_cv_f1", ascending=False))

grids = {
    "Logistic Regression - Lasso": grid_lr_lasso,
    "Logistic Regression - Ridge": grid_lr_ridge,
    "Logistic Regression - ElasticNet": grid_lr_elasticnet,
}

best_model_name = max(
    grids,
    key=lambda name: grids[name].best_score_
)

best_grid = grids[best_model_name]
best_model_lr = best_grid.best_estimator_

print(f"Modèle sélectionné : {best_model_name}")
print(f"F1 moyen en validation croisée : {best_grid.best_score_:.4f}")
print(f"Meilleurs paramètres : {best_grid.best_params_}")
print(f"Meilleur modèle évalué: {best_model_lr}") 

,model,mean_cv_f1
1,Logitic Regression - Ridge,0.764542
0,Logitic Regression - Lasso,0.764135
2,Logitic Regression - ElasticNet,0.764135


Modèle sélectionné : Logistic Regression - Ridge
F1 moyen en validation croisée : 0.7645
Meilleurs paramètres : {'model__C': 100.0, 'model__class_weight': None, 'model__max_iter': 300, 'model__penalty': 'l2', 'model__solver': 'liblinear', 'model__tol': 0.01}
Meilleur modèle évalué: Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age',
                                                   'total_pages_visited']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first'),
                                                  ['country', 'source',
                                                   'new_user'])])),
                ('model',
                 LogisticRegression(C=100.0, max_iter=300, random_state=42,
                                    solver='liblinear', tol=0.01))])


In [25]:
# Optimisation du seuil
tuned_model = TunedThresholdClassifierCV(
    estimator=best_model_lr,
    scoring="f1",
    thresholds=100,
    cv=5,
    n_jobs=-1,
    store_cv_results=True
)

tuned_model.fit(X_train, y_train)

print("Seuil optimal :", tuned_model.best_threshold_)
print("F1 CV estimé :", tuned_model.best_score_)

Seuil optimal : 0.40403904916794664
F1 CV estimé : 0.7709213543941115


#### Les hyperparamètres retenus pour le modèle de Régression Logistique sont les suivants :
- Penalty: 
- C :
- l1_ratio :
- solver :
- class_weight :
- tol :
- seuil de classification :

#### Modèle 3 : XGBoost

In [26]:
# Preprocessing XGBoost

numeric_features = ["age", "total_pages_visited"]
categorical_features = ["country", "source", "new_user"]

categorical_transformer = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [27]:
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()

scale_pos_weight = n_neg / n_pos

print(f"Classe 0 : {n_neg}")
print(f"Classe 1 : {n_pos}")
print(f"scale_pos_weight de référence : {scale_pos_weight:.3f}")

Classe 0 : 220320
Classe 1 : 7342
scale_pos_weight de référence : 30.008


In [28]:
# Gridsearch 1 qui teste l'apprentissage du modèle :
params_xgb_gs1 = {
    "model__learning_rate": [0.01, 0.1, 0.3, 0.5],
    "model__n_estimators": [50, 100, 300, 500],
}

# Gridsearch 2 qui teste la profondeur et le surapprentissage du modèle :
params_xgb_gs2 = {
    "model__max_depth": [1, 3, 5, 7],
    "model__min_child_weight": [1, 3, 5],
    "model__colsample_bytree": [0.5, 0.7, 1.0],
    "model__reg_alpha": [0, 0.5, 1.0],
    "model__subsample": [0.5, 0.7, 1.0],
}

# Gridsearch 3 qui teste l'interaction entre les hyperparamètres du modèle :
params_xgb_gs3 = {
    "model__scale_pos_weight": [
        1.0,
        scale_pos_weight / 2,
        scale_pos_weight,
        scale_pos_weight * 1.5,
    ],
}

In [29]:
model = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    n_jobs=1,
)

pipeline_xgb = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model),
])

scoring = {
    "f1": "f1",
    "recall": "recall",
    "pr_auc": "average_precision",
    "log_loss": "neg_log_loss",
}

In [30]:
grid_xgb_gs1 = GridSearchCV(
    estimator=pipeline_xgb,
    param_grid=params_xgb_gs1,
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)
grid_xgb_gs1.fit(X_train, y_train)

grid_xgb_gs2 = GridSearchCV(
    estimator=pipeline_xgb,
    param_grid=params_xgb_gs2,
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)
grid_xgb_gs2.fit(X_train, y_train)

grid_xgb_gs3 = GridSearchCV(
    estimator=pipeline_xgb,
    param_grid=params_xgb_gs3,
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)
grid_xgb_gs3.fit(X_train, y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         'passthrough',
                                                                         ['age',
                                                                          'total_pages_visited']),
                                                                        ('cat',
                                                                         OneHotEncoder(handle_unknown='ignore',
                                                                                       sparse_output=False),
                                                                         ['country',
                                                                          'source',
                                                                          'new_user'])])),
                                       ('model',
                                        XGBClassifier(base_score=None,
                                                      booster=No...
                                                      multi_strategy=None,
                                                      n_estimators=None,
                                                      n_jobs=1,
                                                      num_parallel_tree=None, ...))]),
             n_jobs=-1,
             param_grid={'model__scale_pos_weight': [1.0,
                                                     np.float64(15.00408608008717),
                                                     np.float64(30.00817216017434),
                                                     np.float64(45.01225824026151)]},
             refit='f1', return_train_score=True,
             scoring={'f1': 'f1', 'log_loss': 'neg_log_loss',
                      'pr_auc': 'average_precision', 'recall': 'recall'})

In [31]:
results_xgb_gs1 = pd.DataFrame(grid_xgb_gs1.cv_results_)
display(results_xgb_gs1.head(2))

results_xgb_gs2 = pd.DataFrame(grid_xgb_gs2.cv_results_)
display(results_xgb_gs2.head(2))

results_xgb_gs3 = pd.DataFrame(grid_xgb_gs3.cv_results_)
display(results_xgb_gs3.head(2))

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__learning_rate,param_model__n_estimators,params,split0_test_f1,split1_test_f1,split2_test_f1,...,mean_test_log_loss,std_test_log_loss,rank_test_log_loss,split0_train_log_loss,split1_train_log_loss,split2_train_log_loss,split3_train_log_loss,split4_train_log_loss,mean_train_log_loss,std_train_log_loss
0,1.079565,0.028431,0.207880,0.013410,0.01,50,"{'model__learning_rate': 0.01, 'model__n_estim...",0.000000,0.000000,0.000000,...,-0.072806,0.000290,16,-0.072371,-0.072566,-0.072245,-0.072299,-0.072491,-0.072394,0.000119
1,1.566241,0.154122,0.292276,0.040472,0.01,100,"{'model__learning_rate': 0.01, 'model__n_estim...",0.699402,0.695427,0.694764,...,-0.056816,0.000392,15,-0.056201,-0.056469,-0.056026,-0.056123,-0.056319,-0.056228,0.000154


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__colsample_bytree,param_model__max_depth,param_model__min_child_weight,param_model__reg_alpha,param_model__subsample,params,...,mean_test_log_loss,std_test_log_loss,rank_test_log_loss,split0_train_log_loss,split1_train_log_loss,split2_train_log_loss,split3_train_log_loss,split4_train_log_loss,mean_train_log_loss,std_train_log_loss
0,1.245234,0.063488,0.155531,0.013191,0.5,1,1,0.0,0.5,"{'model__colsample_bytree': 0.5, 'model__max_d...",...,-0.042894,0.000852,287,-0.042593,-0.042970,-0.042507,-0.042657,-0.042742,-0.042694,0.000158
1,1.177577,0.018071,0.167396,0.011750,0.5,1,1,0.0,0.7,"{'model__colsample_bytree': 0.5, 'model__max_d...",...,-0.042909,0.000817,293,-0.042664,-0.042973,-0.042418,-0.042629,-0.042753,-0.042687,0.000180


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__scale_pos_weight,params,split0_test_f1,split1_test_f1,split2_test_f1,split3_test_f1,...,mean_test_log_loss,std_test_log_loss,rank_test_log_loss,split0_train_log_loss,split1_train_log_loss,split2_train_log_loss,split3_train_log_loss,split4_train_log_loss,mean_train_log_loss,std_train_log_loss
0,1.624417,0.092365,0.267725,0.007635,1.000000,{'model__scale_pos_weight': 1.0},0.750189,0.762871,0.760044,0.751216,...,-0.042535,0.000982,1,-0.037079,-0.037357,-0.037007,-0.036938,-0.037668,-0.037210,0.000270
1,1.899097,0.058186,0.282465,0.012476,15.004086,{'model__scale_pos_weight': 15.00408608008717},0.609818,0.615493,0.608392,0.606935,...,-0.096761,0.002212,2,-0.089929,-0.090791,-0.088920,-0.090170,-0.091366,-0.090235,0.000826


In [32]:
def summarize_grid_xgb(grid, model_name):
    results = pd.DataFrame(grid.cv_results_)

    columns = [
        "param_model__learning_rate",
        "param_model__n_estimators",
        "param_model__max_depth",
        "param_model__min_child_weight",
        "param_model__colsample_bytree",
        "param_model__reg_alpha",
        "param_model__scale_pos_weight",
        "model__subsample",
        "mean_test_f1",
        "std_test_f1",
        "mean_test_recall",
        "mean_test_pr_auc",
        "rank_test_f1",
        "rank_test_pr_auc",
        "rank_test_log_loss"
    ]

    # Certaines colonnes n'existent pas pour toutes les grilles d'hyperparamètres, donc on filtre celles qui existent dans les résultats
    columns = [column for column in columns if column in results.columns]

    print(f"\n===== {model_name} =====")
    print("Meilleurs paramètres :")
    print(grid.best_params_)
    print(f"Meilleur F1 moyen : {grid.best_score_:.4f}")
    print(f"Meilleur PR-AUC moyen : {results.loc[grid.best_index_, 'mean_test_pr_auc']:.4f}")
    print(f"Meilleur Log Loss moyen : {-results.loc[grid.best_index_, 'mean_test_log_loss']:.4f}")
    print(f"Meilleur Recall moyen : {results.loc[grid.best_index_, 'mean_test_recall']:.4f}")

    return results[columns].sort_values("rank_test_f1")


results_xgb_gs1 = summarize_grid_xgb(grid_xgb_gs1, "XGBoost Classifier - Grid Search 1")
results_xgb_gs2 = summarize_grid_xgb(grid_xgb_gs2, "XGBoost Classifier - Grid Search 2")
results_xgb_gs3 = summarize_grid_xgb(grid_xgb_gs3, "XGBoost Classifier - Grid Search 3")

display(results_xgb_gs1.head(10))
display(results_xgb_gs2.head(10))
display(results_xgb_gs3.head(10))


===== XGBoost Classifier - Grid Search 1 =====
Meilleurs paramètres :
{'model__learning_rate': 0.1, 'model__n_estimators': 100}
Meilleur F1 moyen : 0.7619
Meilleur PR-AUC moyen : 0.8417
Meilleur Log Loss moyen : 0.0410
Meilleur Recall moyen : 0.6963

===== XGBoost Classifier - Grid Search 2 =====
Meilleurs paramètres :
{'model__colsample_bytree': 0.5, 'model__max_depth': 3, 'model__min_child_weight': 5, 'model__reg_alpha': 0, 'model__subsample': 1.0}
Meilleur F1 moyen : 0.7645
Meilleur PR-AUC moyen : 0.8434
Meilleur Log Loss moyen : 0.0407
Meilleur Recall moyen : 0.6957

===== XGBoost Classifier - Grid Search 3 =====
Meilleurs paramètres :
{'model__scale_pos_weight': 1.0}
Meilleur F1 moyen : 0.7566
Meilleur PR-AUC moyen : 0.8332
Meilleur Log Loss moyen : 0.0425
Meilleur Recall moyen : 0.6905


,param_model__learning_rate,param_model__n_estimators,mean_test_f1,std_test_f1,mean_test_recall,mean_test_pr_auc,rank_test_f1,rank_test_pr_auc,rank_test_log_loss
5,0.10,100,0.761889,0.005821,0.696269,0.841695,1,3,2
3,0.01,500,0.761702,0.006223,0.692864,0.842543,2,1,1
4,0.10,50,0.761458,0.006736,0.694227,0.842047,3,2,3
8,0.30,50,0.759677,0.003683,0.694499,0.833410,4,9,6
6,0.10,300,0.759577,0.005178,0.692183,0.838531,5,6,4
2,0.01,300,0.758621,0.005218,0.683057,0.841639,6,4,7
9,0.30,100,0.756639,0.005029,0.690549,0.833206,7,10,8
7,0.10,500,0.756381,0.004826,0.690140,0.835851,8,8,5
12,0.50,50,0.750329,0.009086,0.677745,0.809231,9,15,14
13,0.50,100,0.749502,0.009410,0.678425,0.814781,10,13,12


,param_model__max_depth,param_model__min_child_weight,param_model__colsample_bytree,param_model__reg_alpha,mean_test_f1,std_test_f1,mean_test_recall,mean_test_pr_auc,rank_test_f1,rank_test_pr_auc,rank_test_log_loss
47,3,5,0.5,0.0,0.764501,0.003538,0.695724,0.843417,1,2,6
38,3,3,0.5,0.0,0.764420,0.003302,0.696542,0.843508,2,1,3
151,3,3,0.7,1.0,0.764006,0.004485,0.695588,0.842020,3,48,52
44,3,3,0.5,1.0,0.763962,0.003783,0.695452,0.843288,4,6,4
50,3,5,0.5,0.5,0.763609,0.002532,0.694090,0.843189,5,8,8
43,3,3,0.5,1.0,0.763540,0.003776,0.694499,0.842864,6,14,13
35,3,1,0.5,1.0,0.763509,0.003413,0.695044,0.843024,7,11,9
40,3,3,0.5,0.5,0.763301,0.003543,0.692864,0.842997,8,12,12
34,3,1,0.5,1.0,0.763224,0.003827,0.693954,0.842810,9,17,16
31,3,1,0.5,0.5,0.763223,0.004250,0.693681,0.842836,10,16,18


,param_model__scale_pos_weight,mean_test_f1,std_test_f1,mean_test_recall,mean_test_pr_auc,rank_test_f1,rank_test_pr_auc,rank_test_log_loss
0,1.000000,0.756639,0.005029,0.690549,0.833206,1,2,1
1,15.004086,0.610774,0.003151,0.893897,0.834754,2,1,2
2,30.008172,0.521601,0.007550,0.923046,0.833097,3,3,3
3,45.012258,0.466342,0.012964,0.935575,0.831949,4,4,4


In [34]:
# Gridsearch 4 qui teste l'interaction entre les hyperparamètres du modèle :
params_xgb_gs4 = {
    "model__learning_rate": [0.1, 0.3],
    "model__n_estimators": [50, 100, 300],
    "model__max_depth": [1, 3, 5],
    "model__min_child_weight": [3, 5, 7],
    "model__scale_pos_weight": [1, scale_pos_weight],
    "model__colsample_bytree": [0.5, 0.7, 1.0],
    "model__reg_alpha": [0, 0.3, 0.5],
    "model__subsample": [0.5, 1.0],
}

grid_xgb_gs4 = GridSearchCV(
    estimator=pipeline_xgb,
    param_grid=params_xgb_gs4,
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=1,
    return_train_score=True
)
grid_xgb_gs4.fit(X_train, y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         'passthrough',
                                                                         ['age',
                                                                          'total_pages_visited']),
                                                                        ('cat',
                                                                         OneHotEncoder(handle_unknown='ignore',
                                                                                       sparse_output=False),
                                                                         ['country',
                                                                          'source',
                                                                          'new_user'])])),
                                       ('model',
                                        XGBClassifier(base_score=None,
                                                      booster=No...
                         'model__learning_rate': [0.1, 0.3],
                         'model__max_depth': [1, 3, 5],
                         'model__min_child_weight': [3, 5, 7],
                         'model__n_estimators': [50, 100, 300],
                         'model__reg_alpha': [0, 0.3, 0.5],
                         'model__scale_pos_weight': [1,
                                                     np.float64(30.00817216017434)],
                         'model__subsample': [0.5, 1.0]},
             refit='f1', return_train_score=True,
             scoring={'f1': 'f1', 'log_loss': 'neg_log_loss',
                      'pr_auc': 'average_precision', 'recall': 'recall'})

In [35]:
def summarize_grid_xgb(grid, model_name):
    results = pd.DataFrame(grid.cv_results_)

    columns = [
        "param_model__learning_rate",
        "param_model__n_estimators",
        "param_model__max_depth",
        "param_model__min_child_weight",
        "param_model__colsample_bytree",
        "param_model__reg_alpha",
        "param_model__scale_pos_weight",
        "model__subsample",
        "mean_test_f1",
        "std_test_f1",
        "mean_test_recall",
        "mean_test_pr_auc",
        "rank_test_f1",
        "rank_test_pr_auc",
        "rank_test_log_loss"
    ]

    # Certaines colonnes n'existent pas pour tous les modèles
    columns = [column for column in columns if column in results.columns]

    print(f"\n===== {model_name} =====")
    print("Meilleurs paramètres :")
    print(grid.best_params_)
    print(f"Meilleur F1 moyen : {grid.best_score_:.4f}")
    print(f"Meilleur PR-AUC moyen : {results.loc[grid.best_index_, 'mean_test_pr_auc']:.4f}")
    print(f"Meilleur Log Loss moyen : {-results.loc[grid.best_index_, 'mean_test_log_loss']:.4f}")
    print(f"Meilleur Recall moyen : {results.loc[grid.best_index_, 'mean_test_recall']:.4f}")

    return results[columns].sort_values("rank_test_f1")


results_xgb_gs4 = summarize_grid_xgb(grid_xgb_gs4, "XGBoost Classifier - Grid Search 4")

display(results_xgb_gs4.head(10))


===== XGBoost Classifier - Grid Search 4 =====
Meilleurs paramètres :
{'model__colsample_bytree': 0.7, 'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__min_child_weight': 5, 'model__n_estimators': 100, 'model__reg_alpha': 0, 'model__scale_pos_weight': 1, 'model__subsample': 1.0}
Meilleur F1 moyen : 0.7670
Meilleur PR-AUC moyen : 0.8432
Meilleur Log Loss moyen : 0.0408
Meilleur Recall moyen : 0.7044


,param_model__learning_rate,param_model__n_estimators,param_model__max_depth,param_model__min_child_weight,param_model__colsample_bytree,param_model__reg_alpha,param_model__scale_pos_weight,mean_test_f1,std_test_f1,mean_test_recall,mean_test_pr_auc,rank_test_f1,rank_test_pr_auc,rank_test_log_loss
805,0.1,100,3,5,0.7,0.0,1.0,0.767014,0.003424,0.704442,0.843225,1,203,155
809,0.1,100,3,5,0.7,0.3,1.0,0.766851,0.003150,0.703352,0.843204,2,221,163
1548,0.1,50,5,5,1.0,0.0,1.0,0.766813,0.007507,0.707984,0.842474,3,480,350
1556,0.1,50,5,5,1.0,0.5,1.0,0.766563,0.007934,0.707166,0.842536,4,454,422
773,0.1,100,3,3,0.7,0.3,1.0,0.766382,0.005514,0.701308,0.843108,5,252,159
841,0.1,100,3,7,0.7,0.0,1.0,0.766329,0.003558,0.703760,0.843242,6,196,149
1552,0.1,50,5,5,1.0,0.3,1.0,0.766313,0.007452,0.707302,0.842491,7,475,397
1584,0.1,50,5,7,1.0,0.0,1.0,0.766268,0.005867,0.703080,0.842854,8,347,293
849,0.1,100,3,7,0.7,0.5,1.0,0.766241,0.004012,0.702943,0.843117,9,243,171
1592,0.1,50,5,7,1.0,0.5,1.0,0.766214,0.006403,0.703761,0.842786,10,366,329


In [36]:
model_scores = pd.DataFrame({
    "model": ["XGB GS1", "XGB GS2", "XGB GS3", "XGB GS4"],
    "mean_cv_f1": [
        grid_xgb_gs1.best_score_,
        grid_xgb_gs2.best_score_,
        grid_xgb_gs3.best_score_,
        grid_xgb_gs4.best_score_,
    ],
})

display(model_scores.sort_values("mean_cv_f1", ascending=False))

grids = {
    "XGB GS1": grid_xgb_gs1,
    "XGB GS2": grid_xgb_gs2,
    "XGB GS3": grid_xgb_gs3,
    "XGB GS4": grid_xgb_gs4,
}

best_model_name = max(
    grids,
    key=lambda name: grids[name].best_score_
)

best_grid = grids[best_model_name]
best_model_xgb = best_grid.best_estimator_

print(f"Modèle sélectionné : {best_model_name}")
print(f"F1 moyen en validation croisée : {best_grid.best_score_:.4f}")
print(f"Meilleurs paramètres : {best_grid.best_params_}")
print(f"Meilleur modèle évalué: {best_model_xgb}")

,model,mean_cv_f1
3,XGB GS4,0.767014
1,XGB GS2,0.764501
0,XGB GS1,0.761889
2,XGB GS3,0.756639


Modèle sélectionné : XGB GS4
F1 moyen en validation croisée : 0.7670
Meilleurs paramètres : {'model__colsample_bytree': 0.7, 'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__min_child_weight': 5, 'model__n_estimators': 100, 'model__reg_alpha': 0, 'model__scale_pos_weight': 1, 'model__subsample': 1.0}
Meilleur modèle évalué: Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['age',
                                                   'total_pages_visited']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['country', 'source',
                                                   'new_user'])])),
                ('model',
                 XGBClassi

In [37]:
# Optimisation du seuil
tuned_model = TunedThresholdClassifierCV(
    estimator=best_model_xgb,
    scoring="f1",
    thresholds=100,
    cv=5,
    n_jobs=-1,
    store_cv_results=True
)

tuned_model.fit(X_train, y_train)

print("Seuil optimal :", tuned_model.best_threshold_)
print("F1 CV estimé :", tuned_model.best_score_)

Seuil optimal : 0.40339926
F1 CV estimé : 0.7694944007246031


#### Les hyperparamètres retenus pour le modèle de XGBoost sont les suivants :
- learning_rate : 
- max_depth :
- min_child_weight :
- subsample :
- colsample_bytree :
- reg_alpha :
- class_pos_weight :
- seuil de classification :